# Predicción de Churn de Clientes
Este notebook contiene el código para entrenar una red neuronal que predice la probabilidad de que un cliente abandone el servicio (Churn) utilizando TensorFlow y Keras.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

## Carga de Datos
Asegúrate de subir el archivo `WA_Fn-UseC_-Telco-Customer-Churn.csv` a tu entorno de Colab (puedes subirlo a la sección de Archivos a la izquierda).

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## Preprocesamiento de Datos
Prepararemos los datos para ser consumidos por la red neuronal.

In [ ]:
# 1. Eliminar la columna 'customerID' ya que no aporta valor predictivo
if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)

# 2. Convertir 'TotalCharges' a numérico. Hay valores vacíos ' ' que se llenarán con 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# 3. Separar características (X) y objetivo (y)
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Aplicar One-Hot Encoding a las variables categóricas
categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(exclude=['object']).columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# 4. Escalar variables numéricas
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

X.head()

## División en Entrenamiento y Prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño del set de entrenamiento: {X_train.shape}")
print(f"Tamaño del set de prueba: {X_test.shape}")

## Construcción del Modelo (Red Neuronal)

In [ ]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid') # Clasificación binaria
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## Entrenamiento del Modelo con Early Stopping y Pesos Balanceados

In [ ]:
# 1. Calcular pesos de clase (hay más clientes que no hacen churn)
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(weights))
print("Pesos para balancear las clases:", class_weights)

# 2. Configurar Early Stopping para evitar el sobreajuste
# Esto detendrá el entrenamiento si 'val_loss' no mejora en 5 épocas
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 3. Entrenar el modelo
history = model.fit(
    X_train, y_train,
    epochs=100, # Aumentamos las épocas, el early stopping lo detendrá a tiempo
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

## Evaluación del Modelo

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Precisión en el set de prueba: {accuracy:.4f}")

plt.figure(figsize=(10, 4))
plt.plot(history.history['accuracy'], label='Precisión de Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Precisión de Validación')
plt.title('Evolución de la Precisión')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()
plt.show()

## Predicciones y Métricas

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print("Matriz de Confusión:")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.show()

print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

## Guardar el Modelo para Producción
Guardaremos el modelo de red neuronal y los objetos de preprocesamiento (`scaler` y nombres de columnas) para que puedas descargarlos y usarlos en tu PC local.

In [ ]:
import joblib

# 1. Guardar el modelo entrenado de Keras/TensorFlow
model.save('churn_model.h5')

# 2. Guardar el escalador (StandardScaler)
joblib.dump(scaler, 'scaler.pkl')

# 3. Guardar los nombres exactos de las columnas usadas en el entrenamiento
# Esto es vital porque el One-Hot Encoding en producción debe generar las mismas columnas
joblib.dump(list(X.columns), 'model_columns.pkl')

print("¡Archivos guardados correctamente!")
print("- churn_model.h5")
print("- scaler.pkl")
print("- model_columns.pkl")
print("\nPor favor, ve al menú izquierdo de Colab (ícono de carpeta) y descarga estos 3 archivos a tu PC.")